# Chapter 7: Supervised Learning - Regression & Classification
## Notebook 02 - Intermediate: Classification

Classification predicts discrete labels. We cover logistic regression, decision trees, SVMs, and evaluation with ROC curves.

**What you'll learn:**
- Logistic regression from scratch
- Decision boundaries and tree structure
- Support Vector Machines and the kernel trick
- ROC curves, AUC, precision-recall
- Handling imbalanced classes

**Time estimate:** 3.5 hours

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*

## 1. Logistic Regression From Scratch

**Sigmoid:** $\sigma(z) = \frac{1}{1+e^{-z}}$ — outputs probability in (0, 1)

**Update rule:** Gradient descent on cross-entropy loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class LogisticRegressionScratch:
    def __init__(self, lr=0.1, epochs=1000):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = None

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).ravel()
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        n, p = X.shape
        self.w = np.zeros(p)
        self.b = 0.0
        for _ in range(self.epochs):
            logits = X @ self.w + self.b
            probs = self.sigmoid(logits)
            err = probs - y
            self.w -= self.lr * (X.T @ err) / n
            self.b -= self.lr * np.mean(err)
        return self

    def predict_proba(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return self.sigmoid(X @ self.w + self.b)

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X) >= thresh).astype(int)

from sklearn.datasets import make_classification
X, y = make_classification(n_samples=150, n_features=2, n_redundant=0, n_informative=2, random_state=42)
logr = LogisticRegressionScratch(epochs=500)
logr.fit(X, y)
acc = np.mean(logr.predict(X) == y)
print(f'Accuracy (from scratch): {acc:.4f}')

## 2. Decision Boundary Visualization

The decision boundary is where $P(y=1|X) = 0.5$ — a line in 2D.

In [ ]:
def plot_decision_boundary(model, X, y, title='Decision Boundary'):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu_r', levels=20)
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu_r', edgecolors='black', s=50)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title)
    plt.colorbar(label='P(y=1)')
    plt.tight_layout()
    plt.show()

plot_decision_boundary(logr, X, y, 'Logistic Regression: Decision Boundary')

## 3. Decision Trees

Trees split on features to create regions. Each leaf predicts a class.

See `assets/diagrams/decision_tree.svg` for a loan approval example.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_tree(dt, ax=axes[0], filled=True, feature_names=['F1', 'F2'], class_names=['0', '1'])
axes[0].set_title('Decision Tree Structure')

# Decision boundary
h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = dt.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
axes[1].contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu_r')
axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu_r', edgecolors='black', s=50)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_title('Decision Tree: Colored Regions')
plt.tight_layout()
plt.show()

## 4. Support Vector Machines & Kernel Trick

SVM finds the **maximum margin** hyperplane. The **kernel trick** maps data to higher dimensions for nonlinear boundaries.

In [ ]:
from sklearn.svm import SVC

# Linear SVM
svm_linear = SVC(kernel='linear', C=1.0).fit(X, y)

# RBF kernel - nonlinear boundary
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale').fit(X, y)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model, title in [(axes[0], svm_linear, 'SVM Linear'), (axes[1], svm_rbf, 'SVM RBF Kernel')]:
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu_r')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu_r', edgecolors='black', s=50)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

## 5. ROC Curves and AUC

**ROC:** True Positive Rate vs False Positive Rate at different thresholds.

**AUC:** Area under ROC curve — 1.0 = perfect, 0.5 = random.

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
from sklearn.linear_model import LogisticRegression
logr_sk = LogisticRegression(max_iter=1000)
logr_sk.fit(X_train, y_train)
y_proba = logr_sk.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec, prec, 'darkgreen', lw=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Imbalanced Classes: Credit Default

When one class is rare (e.g., defaults), accuracy can be misleading. Use **class_weight**, **oversampling**, or focus on recall/AUC.

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('..') / 'datasets' / 'credit.csv')
X = df.drop(columns=['default']).values
y = df['default'].values
print(f'Default rate: {y.mean():.2%}')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Without class weight
lr_default = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
pred_default = lr_default.predict(X_test_s)

# With class_weight='balanced'
lr_balanced = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train_s, y_train)
pred_balanced = lr_balanced.predict(X_test_s)

print('\nDefault (no weights):')
print(classification_report(y_test, pred_default, target_names=['No default', 'Default']))
print('Balanced class weights:')
print(classification_report(y_test, pred_balanced, target_names=['No default', 'Default']))

### Interactive: Predict default risk

Enter applicant features and get probability of default.

In [ ]:
# Prediction prompt - customer classification
age = 35
income = 50000
debt_ratio = 0.25
credit_score = 720
employment_years = 8

applicant = scaler.transform([[age, income, debt_ratio, credit_score, employment_years]])
prob_default = lr_balanced.predict_proba(applicant)[0, 1]
print(f'Applicant: age={age}, income=${income}, debt_ratio={debt_ratio}, credit_score={credit_score}')
print(f'Predicted default probability: {prob_default:.1%}')
print(f'Recommendation: {"Approve" if prob_default < 0.3 else "Review" if prob_default < 0.6 else "Decline"}')

## 7. Learning Curves

Plot train vs validation score vs sample size — diagnose overfitting/underfitting.

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(max_iter=1000), X_train_s, y_train, cv=5,
    train_sizes=np.linspace(0.2, 1.0, 10)
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'b-o', label='Train')
plt.plot(train_sizes, val_scores.mean(axis=1), 'r-o', label='Validation')
plt.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.2)
plt.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                 val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.2)
plt.xlabel('Training set size')
plt.ylabel('Accuracy')
plt.title('Learning Curves: Credit Default (Logistic Regression)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*